In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram


# Gates and noise
In this notebook we'll use the noisy circuit simulator (Qiskit Aer) to start investigating how noise affects quantum operations.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

In [ ]:
U = np.diag([1, -1, -1, -1])
CZ = UnitaryGate(U, label = 'CZ', num_qubits = 2)

In [ ]:
circ = QuantumCircuit(2)
circ.h([0, 1])
circ.append(CZ, [0, 1])
circ.h([1])

In [ ]:
circ.measure_all()
circ.draw()

Add a depolarizing error to the CZ gate. This is the most basic (Pauli) error that can be applied to a circuit. It means that we, with equal probability, apply any two-qubit Pauli operator to our state (e.g. $XY$, $XZ$, or $IX$, where $I$ is the identity operator that does nothing). A realistic noise model is far more complicated, and generally contains individual probabilities for each of these errors.

In [ ]:
err = depolarizing_error(0.1, 1)
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(err, ['CZ'])
print(noise_model)

Simulate both the noisy and ideal circuits using Aer.

In [ ]:
sim_ideal = AerSimulator()
result_ideal = sim_ideal.run(circ).result()
res_ideal = result_ideal.get_counts()

In [ ]:
sim_noisy = AerSimulator(noise_model = noise_model)
result_noisy = sim_noisy.run(circ).result()
res_noise = result_noisy.get_counts()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (7,3))
plot_histogram(res_ideal, ax = ax[0])
plot_histogram(res_noise, ax = ax[1])
ax[0].set_title("Noise free")
ax[1].set_title("Noisy")

plt.tight_layout()

## 1. What's the state fidelity now?

**Exercise**
- Write a function that calculates the average parity given a dictionary of outcomes.
- Use this to calculate the parity as a function of $\phi$ again to get an idea of the fidelity.

In [ ]:
## If you want to use the subsequent analysis pipeline you
## should complete the following function:

def ave_parity(cts: qiskit.result.counts.Counts) -> float:
    """
    Returns the parity (expectation value of ZZ) given a
    dictionary of counts, cts, where the keys are possible
    measurement outcomes, while the values are how often 
    these outcomes were observed in the simulation. For
    instance:
        cts = {'01': 23, '11': 54}
    """

In [ ]:
from qiskit.circuit import Parameter

In [ ]:
phi = Parameter("ϕ")
circ = QuantumCircuit(2)
circ.h([0, 1])
circ.append(CZ, [0, 1])
circ.h([1])
circ.rx(phi, [0])
circ.measure_all()
circ.draw()

In [ ]:
phis = np.linspace(0, 2*np.pi, 50)
res_ideal = sim_ideal.run(circ, parameter_binds = [{phi: phis}]).result()
res_noisy = sim_noisy.run(circ, parameter_binds = [{phi: phis}]).result()

In [ ]:
type(res_ideal.get_counts()[0])

In [ ]:
[ave_parity(cts) for cts in res_ideal.get_counts()]

In [ ]:
parities_ideal = [ave_parity(cts) for cts in res_ideal.get_counts()]
parities_noisy = [ave_parity(cts) for cts in res_noisy.get_counts()]

plt.plot(phis, parities_ideal, label = "Ideal")
plt.plot(phis, parities_noisy, label = "Noisy")
plt.xlabel(r"$\phi$", fontsize = 12)
plt.ylabel(r"Parity", fontsize = 12)
plt.legend()